<a href="https://colab.research.google.com/github/kyeeun7706-coder/e-waste-location-optimization/blob/main/%EC%A0%84%EC%A2%85%EC%84%A40507_%EB%8F%84%EB%B4%89%EA%B5%AC_%EA%B1%B0%EB%A6%AC%ED%96%89%EB%A0%AC%EB%8D%B0%EC%9D%B4%ED%84%B0_%ED%9B%84%EB%B3%B4%EC%A7%8071%EA%B0%9C_%EB%8B%A4%EB%84%A3%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# [Cell 1] 패키지 설치 (Colab 세션마다 1회 실행)
# ============================================================
!pip install osmnx pyshp pyproj -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.2 MB/s eta 0:00:00


In [2]:
# ============================================================
# [Cell 2] 라이브러리 임포트 + 경로 설정
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, math, time, warnings
import pandas as pd
import networkx as nx
import osmnx as ox
import shapefile
from pyproj import Transformer

warnings.filterwarnings('ignore')

drive.mount("/content/drive")


# ══════════════════════════════════════════════════════
# 0. 경로 설정 (필요 시 수정)
# ══════════════════════════════════════════════════════
BASE_DIR = "/content/drive/MyDrive/전종설/0501"
SLOPE_SHP   = os.path.join(BASE_DIR, "도봉구_집계구별_평균경사도(논문지표응용).shp")
SLOPE_CSV   = os.path.join(BASE_DIR, "도봉구_집계구별_평균경사도(논문지표응용).csv")
HOUSE_CSV   = os.path.join(BASE_DIR, "11_2024년_주택유형별주택.csv")
INDEX_XL    = os.path.join(BASE_DIR, "0414final_index.xlsx")
FACILITY_XL = os.path.join(BASE_DIR, "도봉구_생활환경시설_통합목록_좌표완성(최종).xlsx")

OUT_DIR     = os.path.join(BASE_DIR, "outputs")
OUT_DEMAND  = os.path.join(OUT_DIR, "수요지_최종목록_osmnx.csv")
OUT_SUPPLY  = os.path.join(OUT_DIR, "후보지_최종목록_osmnx.csv")
OUT_MATRIX  = os.path.join(OUT_DIR, "경사도보정_거리행렬_osmnx.csv")
os.makedirs(OUT_DIR, exist_ok=True)

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ══════════════════════════════════════════════════════
# 1. 경사도 보정계수 함수 (논문 Table 6)
# ══════════════════════════════════════════════════════
def get_alpha(slope_deg: float, direction: str = 'up') -> float:
    """
    논문 Table 6 Walking Speed 감소율 기반 보정계수
    α = 1 / (1 - 감소율)  →  체감거리를 늘리는 계수

    Parameters
    ----------
    slope_deg : 경사도 (도, °) — 절댓값으로 전달
    direction : 'up'(오르막) or 'down'(내리막)

    Table 6 Walking Speed 감소율:
                오르막              내리막
      3~5°    -1.46%              -2.64%
      5~7°    -4.89%              -4.04%
      7~9°    -3.04%              -7.34%
      (7도 이상은 수요지/후보지 제외 용도로만 쓰되,
       경로 구간에서 만나면 보정계수 적용)
    """
    if slope_deg < 3:
        return 1.0  # 평지: 보정 없음

    # 감소율 테이블: (최솟값, 최댓값, up_rate, down_rate)
    table = [
        (3, 5, 0.0146, 0.0264),
        (5, 7, 0.0489, 0.0404),
        (7, 9, 0.0304, 0.0734),
        (9, 999, 0.0304, 0.0734),  # 9도 초과는 7~9 값 그대로 사용
    ]
    for lo, hi, up_r, dn_r in table:
        if lo <= slope_deg < hi:
            r = up_r if direction == 'up' else dn_r
            return 1.0 / (1.0 - r)

    return 1.0

def slope_category(deg):
    if pd.isna(deg): return '정보없음'
    if deg < 3:      return '평지(0~3°)'
    elif deg < 5:    return '완경사(3~5°)'
    elif deg < 7:    return '급경사(5~7°)'
    else:            return '험준(7°+)'

def slope_alpha_simple(deg):
    """수요지 집계구 단위 α (오르막 기준, 7도 이상 nan)"""
    if pd.isna(deg): return 1.0
    if deg < 3:      return 1.0
    elif deg < 5:    return get_alpha(4.0, 'up')
    elif deg < 7:    return get_alpha(6.0, 'up')
    else:            return np.nan

In [4]:
# ══════════════════════════════════════════════════════
# 2. 집계구 센트로이드 + 경사도 추출
# ══════════════════════════════════════════════════════
import numpy as np
print("="*60)
print("[STEP 1] 집계구 센트로이드 + 경사도 추출")
print("="*60)

transformer = Transformer.from_crs("EPSG:5186", "EPSG:4326", always_xy=True)
sf = shapefile.Reader(SLOPE_SHP, encodingErrors='replace')

coord_rows = []
for shprec, rec in zip(sf.shapes(), sf.records()):
    pts = shprec.points
    cx  = sum(p[0] for p in pts) / len(pts)
    cy  = sum(p[1] for p in pts) / len(pts)
    lon, lat = transformer.transform(cx, cy)
    oa14 = str(rec[2]).strip()
    oa13 = oa14[:7] + oa14[8:]
    coord_rows.append({'oa_cd': oa13, 'lon': lon, 'lat': lat})

df_coords = pd.DataFrame(coord_rows)

df_slope_csv = pd.read_csv(SLOPE_CSV)
df_slope_csv['oa_cd'] = (df_slope_csv['TOT_OA_CD'].astype(str).str[:7]
                         + df_slope_csv['TOT_OA_CD'].astype(str).str[8:])

df_oa = df_coords.merge(df_slope_csv[['oa_cd','경사도_평균','경사도_등급']], on='oa_cd', how='left')
df_oa['slope_alpha']  = df_oa['경사도_평균'].apply(slope_alpha_simple)
df_oa['경사도_범주'] = df_oa['경사도_평균'].apply(slope_category)
print(f"  → 집계구 좌표+경사도: {len(df_oa)}개")

[STEP 1] 집계구 센트로이드 + 경사도 추출
  → 집계구 좌표+경사도: 603개


In [5]:
# ══════════════════════════════════════════════════════
# 3. 수요지 필터링 (보고서 4개 조건만 적용 — 경사도 제외)
# ══════════════════════════════════════════════════════
print("\n" + "="*60)
print("[STEP 2] 수요지 필터링")
print("="*60)

df_house = pd.read_csv(HOUSE_CSV, header=None, names=['year','oa14','ho_gb','cnt'])
df_house['oa14'] = df_house['oa14'].astype(str).str.strip()
dob_house = df_house[df_house['oa14'].str.startswith('1110')].copy()
dob_house['oa_cd'] = dob_house['oa14'].str[:7] + dob_house['oa14'].str[8:]

all_types = ['ho_gb_001','ho_gb_002','ho_gb_003','ho_gb_004','ho_gb_005','ho_gb_006']
has_type  = set(dob_house[dob_house['ho_gb'].isin(all_types)].dropna(subset=['cnt'])['oa_cd'])
hh_sum    = (dob_house[dob_house['ho_gb'].isin(all_types)]
             .groupby('oa_cd')['cnt'].sum().reset_index(name='total_hh'))
has_hh    = set(hh_sum[hh_sum['total_hh'] > 0]['oa_cd'])

df_idx    = pd.read_excel(INDEX_XL)
dob_idx   = df_idx[df_idx['adstrd_code_se'].astype(str).str.startswith('1132')].copy()
dob_idx['oa_cd'] = dob_idx['oa_cd'].astype(str)
has_pop   = set(dob_idx[dob_idx['total_pop_mean'] > 0]['oa_cd'])
has_eld   = set(dob_idx[dob_idx['elderly_mean']   > 0]['oa_cd'])

shp_idx   = set(df_oa['oa_cd']) & set(dob_idx['oa_cd'])
c4        = shp_idx & has_type & has_hh & has_pop & has_eld  # 215개

# ★ 수정: 경사도 <7° 필터 제거 → 215개 전체 수요지 사용
df_demand = df_oa[df_oa['oa_cd'].isin(c4)].copy().reset_index(drop=True)

# 경사도 7°+ 집계구는 slope_alpha를 1.0으로 채움 (보정 없음 처리)
df_demand['slope_alpha'] = df_demand['slope_alpha'].fillna(1.0)

df_demand = df_demand.merge(hh_sum, on='oa_cd', how='left')
df_demand = df_demand.merge(
    dob_idx[['oa_cd','elderly_mean','total_pop_mean','elderly_ratio']],
    on='oa_cd', how='left')

print(f"  4개 조건 통과 (경사도 조건 없음): {len(df_demand)}개")
print(f"  (경사도 7°+ 포함 {(df_demand['경사도_평균'] >= 7).sum()}개는 slope_alpha=1.0 처리)")

# ══════════════════════════════════════════════════════
# 4. 후보지 정리
# ══════════════════════════════════════════════════════
print("\n" + "="*60)
print("[STEP 3] 후보지 정리")
print("="*60)

df_fac = pd.read_excel(FACILITY_XL).dropna(subset=['위도','경도'])
df_fac = df_fac.drop_duplicates(subset=['시설명/위치','위도','경도']).copy()

lat_arr = df_oa[df_oa['경사도_평균'].notna()]['lat'].values
lon_arr = df_oa[df_oa['경사도_평균'].notna()]['lon'].values
slp_arr = df_oa[df_oa['경사도_평균'].notna()]['경사도_평균'].values

df_fac['경사도_근접집계구'] = df_fac.apply(
    lambda r: slp_arr[((lat_arr-r['위도'])**2+(lon_arr-r['경도'])**2).argmin()], axis=1)
df_fac['경사도_범주'] = df_fac['경사도_근접집계구'].apply(slope_category)

df_supply = df_fac[df_fac['경사도_근접집계구'] < 7].copy().reset_index(drop=True)
print(f"  중복제거+경사도 <7° 후 후보지: {len(df_supply)}개")




[STEP 2] 수요지 필터링
  4개 조건 통과 (경사도 조건 없음): 215개
  (경사도 7°+ 포함 6개는 slope_alpha=1.0 처리)

[STEP 3] 후보지 정리
  중복제거+경사도 <7° 후 후보지: 509개


In [6]:
# ══════════════════════════════════════════════════════
# 5. OSMnx 보행네트워크 + 고도 다운로드
# ══════════════════════════════════════════════════════
print("\n" + "="*60)
print("[STEP 4] 도봉구 보행네트워크 + 고도 다운로드 (OSMnx)")
print("="*60)
print("  ※ 인터넷 연결 필요, 수 분 소요될 수 있습니다...")

import requests
import numpy as np

G = ox.graph_from_place("도봉구, 서울특별시, 대한민국", network_type='walk')
print(f"  → 네트워크 다운로드 완료: {len(G.nodes)}개 노드, {len(G.edges)}개 엣지")

# Open Topo Data API로 노드별 고도 직접 요청 (무료, 배치 100개씩)
def fetch_elevations(nodes_data, batch_size=100):
    """Open Topo Data SRTM30M API로 고도 조회"""
    node_ids = list(nodes_data.keys())
    elevations = {}

    for i in range(0, len(node_ids), batch_size):
        batch = node_ids[i:i+batch_size]
        locations = "|".join(
            f"{nodes_data[n]['y']},{nodes_data[n]['x']}" for n in batch
        )
        url = f"https://api.opentopodata.org/v1/srtm30m?locations={locations}"
        try:
            resp = requests.get(url, timeout=30)
            results = resp.json().get('results', [])
            for node_id, result in zip(batch, results):
                elevations[node_id] = result.get('elevation', 0) or 0
        except Exception as e:
            print(f"  배치 {i//batch_size+1} 오류: {e} → 고도 0으로 처리")
            for node_id in batch:
                elevations[node_id] = 0

        if (i // batch_size + 1) % 5 == 0:
            print(f"  고도 조회 중... {min(i+batch_size, len(node_ids))}/{len(node_ids)}개")
        time.sleep(1)  # API 요청 간격 (과부하 방지)

    return elevations

print("  고도 데이터 조회 중 (노드 수에 따라 수 분 소요)...")
nodes_data = {n: G.nodes[n] for n in G.nodes}
elevations = fetch_elevations(nodes_data)

# 그래프 노드에 고도 주입
nx.set_node_attributes(G, elevations, 'elevation')
print(f"  → 고도 조회 완료")

# 엣지별 경사도 계산
G = ox.elevation.add_edge_grades(G, add_absolute=True)
print(f"  → 엣지 경사도 계산 완료")



[STEP 4] 도봉구 보행네트워크 + 고도 다운로드 (OSMnx)
  ※ 인터넷 연결 필요, 수 분 소요될 수 있습니다...
  → 네트워크 다운로드 완료: 4256개 노드, 12460개 엣지
  고도 데이터 조회 중 (노드 수에 따라 수 분 소요)...
  고도 조회 중... 500/4256개
  고도 조회 중... 1000/4256개
  고도 조회 중... 1500/4256개
  고도 조회 중... 2000/4256개
  고도 조회 중... 2500/4256개
  고도 조회 중... 3000/4256개
  고도 조회 중... 3500/4256개
  고도 조회 중... 4000/4256개
  → 고도 조회 완료
  → 엣지 경사도 계산 완료


In [7]:
# ══════════════════════════════════════════════════════
# 6. 경사도 보정 최단경로 거리 계산
# ══════════════════════════════════════════════════════
print("\n" + "="*60)
print("[STEP 5] 경사도 보정 거리 행렬 계산")
print("="*60)

def slope_adjusted_length(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    """
    각 엣지의 'length'를 경사도 보정한 'slope_length'로 추가.
    grade > 0 → 오르막, grade < 0 → 내리막
    """
    for u, v, k, data in G.edges(data=True, keys=True):
        length = data.get('length', 0)
        grade  = data.get('grade', 0)       # -1 ~ 1 (음수=내리막)
        slope_deg = abs(math.degrees(math.atan(abs(grade))))  # 각도로 변환
        direction = 'up' if grade >= 0 else 'down'
        alpha = get_alpha(slope_deg, direction)
        G[u][v][k]['slope_length'] = length * alpha
    return G

G = slope_adjusted_length(G)
print(f"  → 엣지별 경사도 보정 완료")

def nearest_node(G, lat, lon):
    """좌표에서 가장 가까운 네트워크 노드 반환"""
    return ox.distance.nearest_nodes(G, lon, lat)

# 수요지/후보지 → 가장 가까운 네트워크 노드
print("  수요지 노드 매핑 중...")
demand_nodes = [nearest_node(G, row.lat, row.lon) for _, row in df_demand.iterrows()]

print("  후보지 노드 매핑 중...")
supply_nodes = [nearest_node(G, row['위도'], row['경도']) for _, row in df_supply.iterrows()]

# 거리 행렬 계산
n_d = len(df_demand)
n_s = len(df_supply)
dist_matrix = np.full((n_d, n_s), np.nan)

print(f"  {n_d} × {n_s} = {n_d*n_s:,}쌍 최단경로 계산 중...")
print("  (시간이 걸립니다. 수요지 1개당 후보지 전체 탐색)")

t0 = time.time()
for i, d_node in enumerate(demand_nodes):
    try:
        # 수요지 노드에서 모든 후보지 노드까지 최단경로 (slope_length 기준)
        lengths = nx.single_source_dijkstra_path_length(
            G, d_node, weight='slope_length'
        )
        for j, s_node in enumerate(supply_nodes):
            dist_matrix[i, j] = lengths.get(s_node, np.nan)
    except Exception:
        pass  # 경로 없는 경우 nan 유지

    if (i+1) % 20 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i+1) * (n_d - i - 1)
        print(f"  {i+1}/{n_d} 완료 | 경과 {elapsed:.0f}s | 예상 잔여 {eta:.0f}s")

print(f"\n  → 계산 완료! 총 {time.time()-t0:.1f}초")


[STEP 5] 경사도 보정 거리 행렬 계산
  → 엣지별 경사도 보정 완료
  수요지 노드 매핑 중...
  후보지 노드 매핑 중...
  215 × 509 = 109,435쌍 최단경로 계산 중...
  (시간이 걸립니다. 수요지 1개당 후보지 전체 탐색)
  20/215 완료 | 경과 1s | 예상 잔여 6s
  40/215 완료 | 경과 1s | 예상 잔여 5s
  60/215 완료 | 경과 2s | 예상 잔여 4s
  80/215 완료 | 경과 2s | 예상 잔여 4s
  100/215 완료 | 경과 3s | 예상 잔여 3s
  120/215 완료 | 경과 3s | 예상 잔여 3s
  140/215 완료 | 경과 4s | 예상 잔여 2s
  160/215 완료 | 경과 5s | 예상 잔여 2s
  180/215 완료 | 경과 5s | 예상 잔여 1s
  200/215 완료 | 경과 6s | 예상 잔여 0s

  → 계산 완료! 총 6.0초


In [8]:
# ══════════════════════════════════════════════════════
# 7. 결과 통계 출력
# ══════════════════════════════════════════════════════
valid = dist_matrix[~np.isnan(dist_matrix)]
print(f"\n  거리 행렬 통계 (경로 도달 가능 쌍):")
print(f"    도달 가능: {(~np.isnan(dist_matrix)).sum():,}쌍 / 전체 {n_d*n_s:,}쌍")
print(f"    평균:  {valid.mean():.1f}m")
print(f"    최소:  {valid.min():.1f}m")
print(f"    최대:  {valid.max():.1f}m")
print(f"    500m 이내: {(valid <= 500).mean()*100:.1f}%")

print(f"\n  경사도 범주별 평균거리:")
for cat in ['평지(0~3°)', '완경사(3~5°)', '급경사(5~7°)']:
    mask = (df_demand['경사도_범주'] == cat).values
    if mask.sum() > 0:
        avg = dist_matrix[mask][~np.isnan(dist_matrix[mask])].mean()
        print(f"    {cat}: {mask.sum()}개 수요지, 평균 {avg:.1f}m")


  거리 행렬 통계 (경로 도달 가능 쌍):
    도달 가능: 109,435쌍 / 전체 109,435쌍
    평균:  2254.2m
    최소:  0.0m
    최대:  7000.6m
    500m 이내: 3.9%

  경사도 범주별 평균거리:
    평지(0~3°): 161개 수요지, 평균 2252.1m
    완경사(3~5°): 18개 수요지, 평균 2275.0m
    급경사(5~7°): 4개 수요지, 평균 2152.0m


In [9]:
# ══════════════════════════════════════════════════════
# 8. 저장
# ══════════════════════════════════════════════════════
print("\n" + "="*60)
print("[STEP 6] 저장")
print("="*60)

# ① 수요지 목록
demand_out = df_demand[['oa_cd','lat','lon','경사도_평균','경사도_범주',
                         'slope_alpha','total_hh','elderly_mean','total_pop_mean']].copy()
demand_out.columns = ['집계구코드','위도','경도','경사도_평균(°)','경사도_범주',
                       '보정계수(α)','세대수','고령인구_mean','거주인구_mean']
demand_out.to_csv(OUT_DEMAND, index=False, encoding='utf-8-sig')
print(f"  ① 수요지 목록: {OUT_DEMAND}")

# ② 후보지 목록
df_supply.to_csv(OUT_SUPPLY, index=False, encoding='utf-8-sig')
print(f"  ② 후보지 목록: {OUT_SUPPLY}")

# ③ 거리 행렬
d_ids = df_demand['oa_cd'].tolist()
s_ids = df_supply['시설명/위치'].tolist()
df_matrix = pd.DataFrame(dist_matrix, index=d_ids, columns=s_ids)
df_matrix.index.name = '집계구코드(수요지)'
df_matrix.to_csv(OUT_MATRIX, encoding='utf-8-sig')
print(f"  ③ 거리 행렬 ({n_d}×{n_s}): {OUT_MATRIX}")

print("\n✅ 완료!")



[STEP 6] 저장
  ① 수요지 목록: /content/drive/MyDrive/전종설/0501/outputs/수요지_최종목록_osmnx.csv
  ② 후보지 목록: /content/drive/MyDrive/전종설/0501/outputs/후보지_최종목록_osmnx.csv
  ③ 거리 행렬 (215×509): /content/drive/MyDrive/전종설/0501/outputs/경사도보정_거리행렬_osmnx.csv

✅ 완료!


In [10]:
# ══════════════════════════════════════════════════════
# [0507 수정] 후보지 정리 - 경사도 7° 필터링 제거해서 71개 전체 포함한 파일 생성
# ══════════════════════════════════════════════════════
print("=" * 60)
print("[STEP 3-수정] 후보지 정리 (경사도 필터링 없음)")
print("=" * 60)

df_fac = pd.read_excel(FACILITY_XL).dropna(subset=['위도', '경도'])
df_fac = df_fac.drop_duplicates(subset=['시설명/위치', '위도', '경도']).copy()

lat_arr = df_oa[df_oa['경사도_평균'].notna()]['lat'].values
lon_arr = df_oa[df_oa['경사도_평균'].notna()]['lon'].values
slp_arr = df_oa[df_oa['경사도_평균'].notna()]['경사도_평균'].values

df_fac['경사도_근접집계구'] = df_fac.apply(
    lambda r: slp_arr[((lat_arr - r['위도'])**2 + (lon_arr - r['경도'])**2).argmin()], axis=1
)
df_fac['경사도_범주'] = df_fac['경사도_근접집계구'].apply(slope_category)

# ★ 경사도 필터링 없이 전체 사용
df_supply = df_fac.copy().reset_index(drop=True)
print(f"  중복제거 후 후보지 (필터링 없음): {len(df_supply)}개")

# 확인
suhan = df_supply[df_supply['시설명/위치'].str.strip() == '선한노인전문요양원']
print(f"  선한노인전문요양원 포함 여부: {len(suhan) > 0} (경사도: {suhan['경사도_근접집계구'].values}°)")

# 저장
OUT_SUPPLY_V2 = os.path.join(OUT_DIR, "후보지_최종목록_71개.csv")
df_supply.to_csv(OUT_SUPPLY_V2, index=False, encoding='utf-8-sig')
print(f"  → 저장 완료: {OUT_SUPPLY_V2}")

[STEP 3-수정] 후보지 정리 (경사도 필터링 없음)
  중복제거 후 후보지 (필터링 없음): 513개
  선한노인전문요양원 포함 여부: True (경사도: [8.28]°)
  → 저장 완료: /content/drive/MyDrive/전종설/0501/outputs/후보지_최종목록_71개.csv
